In [1]:
import awkward as ak
import numpy as np
import os
import json
import time
import uproot

from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
from coffea.analysis_tools import PackedSelection

import sys
sys.path.append("..")
from src.processors.Processor import Processor
from src.processors.GenMatch import GenMatch

# Batch test

In [2]:
uproot.open.defaults["xrootd_handler"] = uproot.MultithreadedXRootDSource
_events = NanoEventsFactory.from_root(
    file='/eos/user/d/dfu/samples/ZpToHGamma_M1000/output_NanoAODv9.root',
    treepath='Events', schemaclass=NanoAODSchema
).events()
_events
p=Processor(outdir='./test/', mode='mc_2018_ZpToHGamma', JSONdir='../src/json/')
p.process(_events)

{'mc_2018_ZpToHGamma': {'n_events': 120000.0,
  'triggered': 101951.0,
  'filtered': 101726.0,
  'b-veto': 51477.0,
  'photon': 44443.0,
  'AK8jet': 25684.0,
  '2018_HEM_correction': 24518.0,
  'photon-jet_cleaning': 24264.0}}

In [40]:
s = _events.GenPart[
    (abs(_events.GenPart.pdgId) == 9906663) &
    _events.GenPart.hasFlags(["fromHardProcess", "isLastCopy"])
]

In [46]:
a = ak.Array([
    [True, False],
    [True],
    [False, True, True]
])

b = ak.Array([
    True, False, False
])

In [48]:
print(a&b)

[[True, False], [False], [False, False, False]]


In [51]:
_events = NanoEventsFactory.from_root(
    file='root://cms-xrd-global.cern.ch//store/mc/RunIIAutumn18NanoAODv7/ZpToHGamma_M1400_13TeV_TuneCP5_madgraph-pythia8/NANOAODSIM/Nano02Apr2020_102X_upgrade2018_realistic_v21-v1/250000/9FCD7BF6-406A-F34A-9AA9-7790134908D8.root',
    treepath='Events', schemaclass=NanoAODSchema
).events()

In [57]:
a = _events.GenPart[0]

In [63]:
ak.mask(a, a.pdgId==2)['pt']

<Array [0, None, None, ... None, None, None] type='30 * ?float32[parameters={"__...'>

In [45]:
ak.num(s.children, axis=1)

<Array [1, 1, 1, 1, 0, 1, ... 1, 1, 1, 1, 1, 1] type='42000 * int64'>

In [20]:
ak.num(p.variables['gen_Zp_mass'], axis=1)==None

<Array [False, False, False, ... False, False] type='23185 * ?bool'>

In [32]:
a=ak.Array([None, [1,2]])
a

<Array [None, [1, 2]] type='2 * option[var * int64]'>

In [37]:
c= ak.fill_none(a, [None], axis=-1)

In [38]:
a=ak.Array([[1], [2], [3]])
ak.mask(a, a>1)

<Array [[None], [2], [3]] type='3 * var * ?int64'>

In [33]:
ak.flatten(a, axis=1)

<Array [1, 2] type='2 * int64'>

In [23]:
p.variables['gen_Zp_mass'][~(ak.num(p.variables['gen_Zp_mass'], axis=1)==1)]

<Array [None, None, None, ... None, None, None] type='456 * option[var * ?float3...'>

In [10]:
for i in p.variables.keys():
    if 'Zp_' in i:
        p.variables[i] = ak.flatten(p.variables[i], axis=1)

In [11]:
p.variables

{'AK8jet_mass': <Array [130, 120, 117, 133, ... 135, 138, 100] type='23185 * ?float32[parameters...'>,
 'AK8jet_phi': <Array [-2.75, 0.247, 1.47, ... 1.59, -2.2] type='23185 * ?float32[parameters={"...'>,
 'AK8jet_eta': <Array [-0.317, -1.09, ... -0.987, 0.951] type='23185 * ?float32[parameters={"__...'>,
 'AK8jet_msoftdrop': <Array [124, 120, 114, 132, ... 112, 136, 99.2] type='23185 * ?float32[parameter...'>,
 'AK8jet_pt': <Array [690, 494, 596, 718, ... 628, 686, 606] type='23185 * ?float32[parameters...'>,
 'photon_phi': <Array [0.361, -2.89, -1.63, ... -1.54, 1.48] type='23185 * ?float32[parameters=...'>,
 'photon_mass': <Array [0, 0, 0, 0, 0, 0, ... 0, 0, 0, 0, 0, 0] type='23185 * ?float32'>,
 'photon_pt': <Array [681, 611, 670, 624, ... 653, 691, 326] type='23185 * ?float32[parameters...'>,
 'photon_eta': <Array [-0.178, -0.185, ... -0.653, -0.839] type='23185 * ?float32[parameters={"...'>,
 'event_MET_pt': <Array [16.4, 46.9, 134, ... 120, 15.6, 186] type='23185 * ?float32[para

In [8]:
t = ak.Array(p.variables)

In [17]:
for i in list(p.variables.keys()):
    if 'Zp_' in i:
        del p.variables[i]

In [19]:
ak.Array(a)

<Array [{AK8jet_mass: 130, ... ] type='23185 * {"AK8jet_mass": ?float32[paramete...'>

In [21]:
ak.to_parquet(ak.Array(a), './test.parq')

In [4]:
p.variables

{'AK8jet_mass': <Array [130, 120, 117, 133, ... 135, 138, 100] type='23185 * ?float32[parameters...'>,
 'AK8jet_pt': <Array [690, 494, 596, 718, ... 628, 686, 606] type='23185 * ?float32[parameters...'>,
 'AK8jet_phi': <Array [-2.75, 0.247, 1.47, ... 1.59, -2.2] type='23185 * ?float32[parameters={"...'>,
 'AK8jet_eta': <Array [-0.317, -1.09, ... -0.987, 0.951] type='23185 * ?float32[parameters={"__...'>,
 'AK8jet_msoftdrop': <Array [124, 120, 114, 132, ... 112, 136, 99.2] type='23185 * ?float32[parameter...'>,
 'photon_mass': <Array [0, 0, 0, 0, 0, 0, ... 0, 0, 0, 0, 0, 0] type='23185 * ?float32'>,
 'photon_pt': <Array [681, 611, 670, 624, ... 653, 691, 326] type='23185 * ?float32[parameters...'>,
 'photon_phi': <Array [0.361, -2.89, -1.63, ... -1.54, 1.48] type='23185 * ?float32[parameters=...'>,
 'photon_eta': <Array [-0.178, -0.185, ... -0.653, -0.839] type='23185 * ?float32[parameters={"...'>,
 'event_MET_pt': <Array [16.4, 46.9, 134, ... 120, 15.6, 186] type='23185 * ?float32[para

In [12]:
ak.sum(~ak.any(_events.GenPart[abs(_events.GenPart.pdgId)==22].hasFlags(['isPrompt', 'isLastCopy']), axis=1))

3455

In [23]:
_events.Photon.electronVeto

<Array [[True, True], [True, ... [], [True]] type='1854856 * var * bool[paramete...'>

In [19]:
_events.HLT.fields

['AK8PFJet360_TrimMass30',
 'AK8PFHT700_TrimR0p1PT0p03Mass50',
 'AK8PFHT650_TrimR0p1PT0p03Mass50',
 'AK8PFHT600_TrimR0p1PT0p03Mass50_BTagCSV_p20',
 'CaloJet500_NoJetID',
 'Dimuon13_PsiPrime',
 'Dimuon13_Upsilon',
 'Dimuon20_Jpsi',
 'DoubleEle24_22_eta2p1_WPLoose_Gsf',
 'DoubleEle33_CaloIdL',
 'DoubleEle33_CaloIdL_MW',
 'DoubleEle33_CaloIdL_GsfTrkIdVL_MW',
 'DoubleEle33_CaloIdL_GsfTrkIdVL',
 'DoubleEle37_Ele27_CaloIdL_GsfTrkIdVL',
 'DoubleMediumIsoPFTau32_Trk1_eta2p1_Reg',
 'DoubleMediumIsoPFTau35_Trk1_eta2p1_Reg',
 'DoubleMediumIsoPFTau40_Trk1_eta2p1_Reg',
 'DoubleMu33NoFiltersNoVtx',
 'DoubleMu38NoFiltersNoVtx',
 'DoubleMu23NoFiltersNoVtxDisplaced',
 'DoubleMu28NoFiltersNoVtxDisplaced',
 'DoubleMu4_3_Bs',
 'DoubleMu4_3_Jpsi_Displaced',
 'DoubleMu4_JpsiTrk_Displaced',
 'DoubleMu4_LowMassNonResonantTrk_Displaced',
 'DoubleMu4_PsiPrimeTrk_Displaced',
 'Mu7p5_L2Mu2_Jpsi',
 'Mu7p5_L2Mu2_Upsilon',
 'Mu7p5_Track2_Jpsi',
 'Mu7p5_Track3p5_Jpsi',
 'Mu7p5_Track7_Jpsi',
 'Mu7p5_Track2_Upsilon',
 

In [10]:
ak.sum(ak.num(_events.Electron)>0)

15452

In [6]:
heaviest_jet = ak.flatten(p.object['AK8jet'][ak.argmax(p.object['AK8jet'].msoftdrop, axis=1, keepdims=True)], axis=1)
leading_photon = ak.flatten(p.object['photon'][ak.argmax(p.object['photon'].pt, axis=1, keepdims=True)], axis=1)
p.object['photon-jet'] = heaviest_jet + leading_photon
delta_phi = heaviest_jet.phi - leading_photon.phi
delta_phi = ak.min([delta_phi, 2 * np.pi - delta_phi], axis=0)
final_cut = p.pass_cut(cutName='photon-jet_delta_phi', cut=(delta_phi>2.9), final=True)

In [11]:
p.object['AK8jet'][ak.argmax(p.object['AK8jet'].msoftdrop, axis=1, keepdims=True)]

<FatJetArray [None, None, None, ... None, None, None] type='303 * option[var * ?...'>

In [4]:
p.cuts.names

['photon-jet_delta_phi']

In [3]:
g = GenMatch()
d=g.ZpToHGamma(_events)
d

{'ZpToHGamma': array([ True,  True,  True, ...,  True,  True,  True]),
 'ZpToH(WW)Gamma': array([False, False, False, ..., False, False, False]),
 'ZpToH(bb)Gamma': array([False,  True,  True, ..., False,  True,  True]),
 'ZpToH(ZZ)Gamma': array([False, False, False, ..., False, False, False]),
 'ZpToH(tautau)Gamma': array([ True, False, False, ..., False, False, False]),
 'ZpToH(gammagamma)Gamma': array([False, False, False, ..., False, False, False]),
 'gen_HWW_decay_mode': <Array [None, None, None, ... None, None, None] type='33000 * ?int64'>}

In [41]:
re = d['gen_HWW_decay_mode'][d['ZpToH(WW)Gamma']]

In [44]:
ak.sum(re==32)/len(re), ak.sum(re>16)/len(re)

(0.45201056648077487, 0.8982976225418257)

In [8]:
ak.sum(d['ZpToH(WW)Gamma'])

6814

In [9]:
g.gen_cuts.names

['N_Zp == 1',
 'Zp to H Gamma',
 'H to 2 childs',
 'H to WW',
 'H to bb',
 'H to ZZ',
 'H to tautau',
 'H to gammagamma',
 'WW to 4 childs']

In [39]:
ak.sum(g.gen_cuts.all('Zp to H Gamma', 'N_Zp == 1')), ak.sum(g.gen_cuts.all('Zp to H Gamma', 'H to WW', 'N_Zp == 1')), ak.sum(g.gen_cuts.all('H to WW'))

(32297, 6814, 6956)

In [18]:
t=_events.GenPart[  # shape: (event, H_childs)
    (abs(_events.GenPart.pdgId) == g.PDGID['W+']) &
    _events.GenPart.hasFlags(['fromHardProcess', 'isLastCopy'])
]

In [11]:
ak.sum(d['ZpToHGamma'])

0

In [87]:
ak.mean(ak.flatten(t[t.pdgId==22], axis=1).hasFlags(['fromHardProcess', 'isLastCopy']))

1.0

In [11]:
ak.max((_events.GenPart.pdgId == g.PDGID["Zp"]) & _events.GenPart.hasFlags(g.GEN_FLAGS), axis=1)

<Array [True, True, True, ... True, True, True] type='50000 * ?bool'>

# Parallel computing

In [11]:
cutflow = {}
t0 = time.time()

In [12]:
def parallelProcess(samples, name):
    global cutflow, t0
    cutflow[name] = processor.run_uproot_job(
        fileset={name: samples[name]},
        treename="Events",
        processor_instance=Processor(outdir=f'../output/{name}'),
        executor=processor.futures_executor,
        executor_args={"schema": NanoAODSchema, "workers": 48}, # running on $workers cpu cores
    )
    cutflow['time_'+name] = (time.time() - t0)/60
    with open('../output/cutflow.txt', 'w') as file:
        json.dump(cutflow, file)

parallelProcess(samples=samples, name='ZpToHGamma')
parallelProcess(samples=samples, name='GJets')
#parallelProcess(samples=samples, name='WJetsToQQ')
#parallelProcess(samples=samples, name='ZJetsToQQ')
#parallelProcess(samples=samples, name='QCD')



Processing:   0%|          | 0/50 [00:00<?, ?chunk/s]

/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1962: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("2.0.0"):
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1962: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("2.0.0"):
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarn

Preprocessing:   0%|          | 0/147 [00:00<?, ?file/s]

Processing:   0%|          | 0/521 [00:00<?, ?chunk/s]

/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 302 jobs, will wait for 97 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 96 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 95 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 93 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 92 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/